In [ ]:
import os
import subprocess
from pathlib import Path
import re

DATA_DIR = Path("/home/swnh/pgc/datasets/nuscenes/v1.0-mini/ply/bin")
ENCODER_BIN = Path("/home/swnh/gpcc/build/sandbox/predgeom")
OUT_BIN = Path("/tmp/test_out.bin")

SCALE = 1000
GROUP = 2
QP = 32

scene_averages = {}
total_size = 0
total_bpp = 0.0
total_frames = 0

print(f"Starting evaluation of mini set targeting: {ENCODER_BIN.name}")
for scene_dir in sorted(DATA_DIR.iterdir()):
    if not scene_dir.is_dir():
        continue
        
    scene_name = scene_dir.name
    scene_sizes = []
    scene_bpps = []
    # Iterate over ply files
    for ply_file in sorted(scene_dir.glob("*.ply")):
        # Fixed Error 1: Convert integers to strings
        # Fixed Error 2: Use '--groups' instead of '--group'
        cmd = [str(ENCODER_BIN), 
               "--input", str(ply_file), 
               "--scale", str(SCALE), 
               "--groups", str(GROUP),
               "--flat-qp", str(QP)]
        try:
            result = subprocess.run(cmd, capture_output=True, text=True, check=True)
            match = re.search(r"Payload size:\s+(\d+)\s+B\s+\(([\d.]+)\s+bpp\)", result.stdout)
            if match:
                scene_sizes.append(int(match.group(1)))
                scene_bpps.append(float(match.group(2)))
        except subprocess.CalledProcessError:
            pass
            
    if scene_sizes:
        avg_size = sum(scene_sizes) / len(scene_sizes)
        avg_bpp = sum(scene_bpps) / len(scene_bpps)
        scene_averages[scene_name] = avg_size
        total_size += sum(scene_sizes)
        total_bpp += sum(scene_bpps)
        total_frames += len(scene_sizes)
        print(f"Scene: {scene_name} - Avg Payload: {avg_size:8.2f} B, Avg bpp: {avg_bpp:5.2f} ({len(scene_sizes):2d} frames)")

if total_frames > 0:
    overall_avg = total_size / total_frames
    overall_avg_bpp = total_bpp / total_frames
    print("-" * 50)
    print(f"Overall Average Payload Size across {len(scene_averages)} scenes ({total_frames} frames): {overall_avg:.2f} B")
    print(f"Overall Avg Payload: {overall_avg:.2f} B, Overall Avg bpp: {overall_avg_bpp:.2f}")
else:
    print("No frames processed.")

Starting evaluation of mini set targeting: predgeom
Scene: scene-0061 - Avg Payload: 73392.72 B, Avg bpp: 16.91 (39 frames)
Scene: scene-0103 - Avg Payload: 64391.57 B, Avg bpp: 14.83 (40 frames)
Scene: scene-0553 - Avg Payload: 58038.15 B, Avg bpp: 13.39 (41 frames)
Scene: scene-0655 - Avg Payload: 65770.76 B, Avg bpp: 15.16 (41 frames)
Scene: scene-0757 - Avg Payload: 67133.95 B, Avg bpp: 15.47 (41 frames)
Scene: scene-0796 - Avg Payload: 80926.45 B, Avg bpp: 18.64 (40 frames)
Scene: scene-0916 - Avg Payload: 77184.88 B, Avg bpp: 17.79 (41 frames)
Scene: scene-1077 - Avg Payload: 78004.22 B, Avg bpp: 17.97 (41 frames)
Scene: scene-1094 - Avg Payload: 77902.95 B, Avg bpp: 17.95 (40 frames)
Scene: scene-1100 - Avg Payload: 78392.57 B, Avg bpp: 18.07 (40 frames)
--------------------------------------------------
Overall Average Payload Size across 10 scenes (404 frames): 72074.92 B
Overall Avg Payload: 72074.92 B, Overall Avg bpp: 16.61


In [ ]:
import os
import subprocess
from pathlib import Path
import re

DATA_DIR = Path("/home/swnh/pgc/datasets/nuscenes/v1.0-mini/ply/bin")
ENCODER_BIN_MVP = Path("/home/swnh/gpcc/mvp/build/app")
OUT_BIN = Path("/tmp/test_out.bin")

scene_averages = {}
total_size = 0
total_frames = 0

print(f"Starting evaluation of MVP set targeting: {ENCODER_BIN_MVP.name}")
for scene_dir in sorted(DATA_DIR.iterdir()):
    if not scene_dir.is_dir():
        continue
        
    scene_name = scene_dir.name
    scene_sizes = []
    
    # Iterate over ply files
    for ply_file in sorted(scene_dir.glob("*.ply")):
        cmd = [str(ENCODER_BIN_MVP), "--input", str(ply_file), "--output", str(OUT_BIN)]
        try:
            result = subprocess.run(cmd, capture_output=True, text=True, check=True)
            match = re.search(r"bitstream size\s+(\d+)\s+B", result.stdout)
            if match:
                scene_sizes.append(int(match.group(1)))
        except subprocess.CalledProcessError:
            pass
            
    if scene_sizes:
        avg_size = sum(scene_sizes) / len(scene_sizes)
        scene_averages[scene_name] = avg_size
        total_size += sum(scene_sizes)
        total_frames += len(scene_sizes)
        print(f"Scene: {scene_name} - Average Payload Size: {avg_size:8.2f} B ({len(scene_sizes):2d} frames)")

if total_frames > 0:
    overall_avg = total_size / total_frames
    print("-" * 50)
    print(f"Overall Average Payload Size across {len(scene_averages)} scenes ({total_frames} frames): {overall_avg:.2f} B")
else:
    print("No frames processed.")
